# 01 — Transformer Efficiency Landscape

The standard transformer's O(n²) attention complexity drives demand for efficient alternatives.

## Quadratic Attention Cost and Why Alternatives Emerged

The standard self-attention operation has two cost components:
- **Compute**: O(n²d) — the QK^T dot product over all n×n token pairs
- **Memory**: O(n²) — storing the full attention matrix

For n=1000 tokens, this is manageable. For n=100,000 (long documents, audio, genomics), the n² term becomes prohibitive: a 100k-token sequence requires ~40GB just for the float16 attention matrix.

**The quadratic wall in practice**:
- GPT-3 (2048 context, 96 layers): manageable
- Claude 2 (100k context): requires sliding window + memory optimisations
- DNA sequence modelling (3B base pairs): standard attention is impossible

**Why alternatives emerged**:
1. *Long-context language modelling*: legal documents, books, codebases
2. *Audio and time series*: high sampling rates produce very long sequences
3. *Biology*: protein sequences (1000s), genomics (millions of bases)
4. *Inference efficiency*: KV cache grows linearly with context; can exhaust GPU memory

Approaches fall into three families: **linear attention** (kernel trick), **SSMs** (recurrent structure), and **hybrid architectures** (SSM + attention).

In [ ]:
# Attention complexity profiling
import numpy as np
import matplotlib.pyplot as plt

seq_lengths = [128, 256, 512, 1024, 2048, 4096, 8192, 16384, 32768]
d_model = 512

# Memory in bytes (float16 = 2 bytes)
def attention_memory_gb(n, dtype_bytes=2):
    """Memory for ONE attention head's QK^T matrix."""
    return n * n * dtype_bytes / 1e9

def attention_flops(n, d, n_heads=8):
    """FLOPs for multi-head attention."""
    # QK^T: n*d_head per head, n heads
    d_head = d // n_heads
    return 2 * n * n * d_head * n_heads

def linear_attention_flops(n, d, n_heads=8):
    """FLOPs for linear attention: O(nd^2) instead of O(n^2d)."""
    d_head = d // n_heads
    return 2 * n * d_head * d_head * n_heads  # kernel trick: (KV) first

mem = [attention_memory_gb(n) * 8 for n in seq_lengths]  # 8 heads
flops_std = [attention_flops(n, d_model) for n in seq_lengths]
flops_lin = [linear_attention_flops(n, d_model) for n in seq_lengths]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.loglog(seq_lengths, mem, 'o-', color='tomato', label='Standard attention')
ax1.axhline(40, color='gray', linestyle='--', label='40 GB GPU')
ax1.axhline(80, color='black', linestyle='--', label='80 GB H100')
ax1.set_xlabel('Sequence length')
ax1.set_ylabel('Attention memory (GB)')
ax1.set_title('Attention Memory vs Sequence Length')
ax1.legend()

ax2.loglog(seq_lengths, flops_std, 'o-', label='Standard O(n²d)', color='tomato')
ax2.loglog(seq_lengths, flops_lin, 's--', label='Linear O(nd²)', color='steelblue')
ax2.set_xlabel('Sequence length')
ax2.set_ylabel('FLOPs')
ax2.set_title('Compute: Standard vs Linear Attention')
ax2.legend()

plt.tight_layout()
plt.savefig('/tmp/attention_complexity.png', dpi=100, bbox_inches='tight')
plt.show()

print('Crossover point (linear cheaper than standard):')
for i, n in enumerate(seq_lengths):
    ratio = flops_std[i] / flops_lin[i]
    marker = '<-- linear wins' if ratio > 1 else ''
    print(f'  n={n:6d}: std/linear = {ratio:.1f} {marker}')